# পাঠ ১০ - প্রোডাকশনে AI এজেন্ট

এই পাঠে আপনি Microsoft Agent Framework (Python) ব্যবহার করে AI এজেন্টদের **প্রোডাকশন প্যাটার্ন** শিখবেন। আমরা আলোচনা করব:

- **পর্যবেক্ষণযোগ্যতা** — এজেন্ট ইন্টারঅ্যাকশনে টাইমিং এবং লগিং যোগ করা
- **মূল্যায়ন** — উত্তর মান নির্ধারের জন্য একটি মূল্যায়নকারী এজেন্ট ব্যবহার
- **খরচ ব্যবস্থাপনা** — টোকেন অপ্টিমাইজেশন এবং মডেল নির্বাচন কৌশল

এই দৃশ্যপট হলো একটি **ট্রাভেল এজেন্ট** যা ব্যবহারকারীদের ভ্রমণ পরিকল্পনায় সাহায্য করে, যার উপরে পর্যবেক্ষণ এবং মূল্যায়ন স্তর সংযুক্ত। 


## সেটআপ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## উৎপাদন বিবেচ্য বিষয়সমূহ

প্রোটোটাইপ থেকে উৎপাদনে AI এজেন্ট স্থানান্তর করার সময় তিনটি প্রধান স্তম্ভে মনোযোগ দেওয়া প্রয়োজন:

1. **পর্যবেক্ষণযোগ্যতা** — এজেন্ট কী করছে, কত সময় নিচ্ছে, এবং কোন টুলগুলো কল করছে তা দেখতে হবে। ট্রেসিং এবং লগিং ছাড়া উৎপাদন সমস্যাগুলি ডিবাগ করা প্রায় অসম্ভব।

2. **মূল্যায়ন** — স্বয়ংক্রিয় মান যাচাই নিশ্চিত করে যে এজেন্টের প্রতিক্রিয়াগুলি সময়ের সাথে সঠিক, সম্পূর্ণ এবং সহায়ক থাকে। একটি মূল্যায়নকারী এজেন্ট সংজ্ঞায়িত মানদণ্ডের বিরুদ্ধে প্রতিক্রিয়াগুলিকে স্কোর করতে পারে।

3. **খরচ ব্যবস্থাপনা** — টোকেন ব্যবহার সরাসরি খরচে প্রভাব ফেলে। প্রম্পট অপ্টিমাইজেশন, মডেল নির্বাচন, এবং ক্যাশিং এর মতো কৌশলগুলি খরচ নিয়ন্ত্রণে রাখতে সাহায্য করে, মান হারাই ছাড়াই।


## একটি পর্যবেক্ষণযোগ্য এজেন্ট তৈরি করা

আমরা ট্রাভেল টুলস সংজ্ঞায়িত করি এবং এজেন্ট কলটিকে টাইমিংয়ের সাথে র‍্যাপ করি যাতে আমরা বিলম্বতার উপর নজর রাখতে পারি। প্রোডাকশন ক্ষেত্রে আপনি OpenTelemetry বা সমমানের ট্রেসিং ব্যাকএন্ডের সঙ্গে একীভূত করবেন।


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## মূল্যায়ন প্যাটার্ন

একটি সাধারণ উৎপাদন প্যাটার্ন হলো দ্বিতীয় একটি এজেন্টকে **মূল্যায়নকারী** হিসেবে ব্যবহার করা। মূল্যায়নকারী প্রাথমিক এজেন্টের প্রতিক্রিয়াকে পূর্ণতা, সঠিকতা, এবং সহায়কতার মতো পূর্বনির্ধারিত মাপকাঠির বিরুদ্ধে স্কোর করে।

এটি সক্ষম করে:
- ব্যবহারকারীদের কাছে প্রতিক্রিয়া পৌঁছানোর আগে স্বয়ংক্রিয় গুণগত গেট
- প্রম্পট বা মডেল পরিবর্তনের সময় রিগ্রেশন সনাক্তকরণ
- সময়ের সাথে এজেন্টের পারফরম্যান্সের নিরবিচ্ছিন্ন পর্যবেক্ষণ


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## খরচ ব্যবস্থাপনার কৌশলসমূহ

উৎপাদন AI এজেন্টের জন্য খরচ নিয়ন্ত্রণ গুরুত্বপূর্ণ। এখানে মূল কৌশলসমূহ দেওয়া হলো:

| কৌশল | বর্ণনা |
|---|---|
| **প্রম্পট অপটিমাইজেশন** | সিস্টেম নির্দেশনাগুলো সংক্ষিপ্ত রাখুন। ইনপুট টোকেন কমাতে অতিরিক্ত প্রাসঙ্গিকতা সরান। |
    "| **মডেল নির্বাচন** | সহজ কাজ যেমন শ্রেণীবিভাগ বা তথ্য আহরণের জন্য ছোট, সস্তা মডেলগুলি (যেমন GPT-5-mini) ব্যবহার করুন এবং জটিল যুক্তির জন্য বড় মডেল সংরক্ষণ করুন। |\n",
| **ক্যাশিং** | ফেরত পাওয়া টুল ফলাফল এবং প্রায়ই ব্যবহৃত প্রশ্নগুলিকে ক্যাশ করুন যাতে পুনরাবৃত্ত API কল এড়ানো যায়। |
| **টোকেন বাজেট** | অপ্রত্যাশিতভাবে দীর্ঘ উত্তর প্রতিরোধে `max_tokens` সীমা নির্ধারণ করুন। |
| **ব্যাচিং** | যেখানে সম্ভব একাধিক ব্যবহারকারীর প্রশ্ন একক API কল-এ একত্র করুন। |

বাস্তবে, একটি স্তরকৃত পদ্ধতি ভাল কাজ করে: সরল অনুরোধগুলোকে দ্রুত, সস্তা মডেলের কাছে পাঠান এবং শুধুমাত্র জটিল প্রশ্নগুলোকে আরও সক্ষম মডেলে উন্নীত করুন।


## সারসংক্ষেপ

এই পাঠে আপনি শিখেছেন কীভাবে:

১. **এজেন্ট ইন্টারঅ্যাকশনে পর্যবেক্ষণ যোগ করা**, সময় এবং লগিং এর মাধ্যমে, যা ট্রেসিং এবং মনিটরিং-এর জন্য ভিত্তি তৈরি করে।
২. **এজেন্টের প্রতিক্রিয়া স্বয়ংক্রিয়ভাবে মূল্যায়ন করা** একটি ইভ্যালুয়েটর এজেন্ট ব্যবহার করে যা সম্পূর্ণতা, সঠিকতা এবং সহায়তার স্কোর নির্ধারণ করে।
৩. **ব্যয় পরিচালনা করা** প্রম্পট অপটিমাইজেশন, মডেল নির্বাচন, ক্যাশিং, এবং টোকেন বাজেটের মাধ্যমে।

এই প্রোডাকশন প্যাটার্নগুলো নিশ্চিত করে আপনার AI এজেন্টগুলি বিশ্বস্ত, পরিমাপযোগ্য, এবং স্কেলে খরচ-সাশ্রয়ী হয়।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
